# Gradient Descent Algorithm - Starter Notebook
Demonstrates batch, stochastic, and mini-batch gradient descent for optimising a simple linear model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
np.random.seed(42)
n = 100
X = np.random.randn(n, 1)
true_w, true_b = 3.5, -1.2
y = true_w * X.ravel() + true_b + np.random.randn(n) * 0.5

def mse(X, y, w, b):
    pred = X.ravel() * w + b
    return np.mean((pred - y) ** 2)

print(f'True weights: w={true_w}, b={true_b}')

In [ ]:
# Batch Gradient Descent
def batch_gd(X, y, lr=0.05, epochs=100):
    w, b = 0.0, 0.0
    n = len(y)
    history = []
    for _ in range(epochs):
        pred = X.ravel() * w + b
        err = pred - y
        dw = (2/n) * np.dot(err, X.ravel())
        db = (2/n) * np.sum(err)
        w -= lr * dw
        b -= lr * db
        history.append(mse(X, y, w, b))
    return w, b, history

# Stochastic Gradient Descent
def sgd(X, y, lr=0.01, epochs=100):
    w, b = 0.0, 0.0
    history = []
    n = len(y)
    for _ in range(epochs):
        idx = np.random.permutation(n)
        for i in idx:
            xi, yi = X[i, 0], y[i]
            pred = xi * w + b
            err = pred - yi
            w -= lr * 2 * err * xi
            b -= lr * 2 * err
        history.append(mse(X, y, w, b))
    return w, b, history

# Mini-batch Gradient Descent
def mini_batch_gd(X, y, lr=0.02, epochs=100, batch_size=16):
    w, b = 0.0, 0.0
    n = len(y)
    history = []
    for _ in range(epochs):
        idx = np.random.permutation(n)
        for start in range(0, n, batch_size):
            batch = idx[start:start+batch_size]
            Xb, yb = X[batch, 0], y[batch]
            pred = Xb * w + b
            err = pred - yb
            w -= lr * (2/len(batch)) * np.dot(err, Xb)
            b -= lr * (2/len(batch)) * np.sum(err)
        history.append(mse(X, y, w, b))
    return w, b, history

w_b, b_b, hist_b = batch_gd(X, y)
w_s, b_s, hist_s = sgd(X, y)
w_m, b_m, hist_m = mini_batch_gd(X, y)

print(f'Batch GD:      w={w_b:.4f}, b={b_b:.4f}')
print(f'SGD:           w={w_s:.4f}, b={b_s:.4f}')
print(f'Mini-batch GD: w={w_m:.4f}, b={b_m:.4f}')

In [ ]:
# Convergence curves
plt.figure(figsize=(8, 4))
plt.plot(hist_b, label='Batch GD')
plt.plot(hist_s, label='SGD')
plt.plot(hist_m, label='Mini-batch GD')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Gradient Descent Variants: Convergence')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Loss surface contour + trajectory (Batch GD)
def trace_batch_gd(X, y, lr=0.05, epochs=30):
    w, b = -2.0, 3.0
    path = [(w, b)]
    n = len(y)
    for _ in range(epochs):
        pred = X.ravel() * w + b
        err = pred - y
        w -= lr * (2/n) * np.dot(err, X.ravel())
        b -= lr * (2/n) * np.sum(err)
        path.append((w, b))
    return path

ws = np.linspace(-3, 6, 100)
bs = np.linspace(-4, 4, 100)
W, B = np.meshgrid(ws, bs)
Z = np.array([[mse(X, y, w_, b_) for w_ in ws] for b_ in bs])

path = trace_batch_gd(X, y)
pw, pb = zip(*path)

plt.figure(figsize=(7, 5))
plt.contourf(W, B, Z, levels=30, cmap='viridis')
plt.colorbar(label='MSE')
plt.plot(pw, pb, 'r-o', markersize=4, label='GD trajectory')
plt.scatter([true_w], [true_b], c='yellow', s=100, zorder=5, marker='*', label='True optimum')
plt.xlabel('w')
plt.ylabel('b')
plt.title('Loss Surface and Gradient Descent Trajectory')
plt.legend()
plt.tight_layout()
plt.show()